In [1]:
import mediapy as media
import numpy as np
import sys
import jax
import jax.numpy as jp
from mujoco_playground._src import mjx_env
import mujoco
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from pathlib import Path
from mujoco import mjx
from ml_collections import config_dict
REPO = Path.cwd().parent          # training/ -> robotic-craftsman/
sys.path.insert(0, str(REPO))
from arm import ROOT, NPZ_PATH, XML_PATH, Calibration, HandPose, MjcArm, DepthScale, ArmIK, STServo, FingerBoard

In [6]:
CLIPS = sorted((ROOT / "refs").glob("*.npz"))
Q_REF = np.load(CLIPS[0])["q_ref"]

In [ ]:
def default_config():
       return config_dict.create(ctrl_dt=CTRL_DT, sim_dt=SIM_DT, episode_length=episode_length)
    

class HandTraining(mjx_env.MjxEnv):
    
    idx_list: list
    run_result: list

    
    def __init__(self, config):
        self.POLICY_ACTS = ["a_elbow_pitch", "a_forearm_roll", "a_wrist_tilt", "a_index_mcp", "a_thumb_mcp", "a_thumb_pip"]
        self.sim = MjcArm()
        self.Q_REF = jp.array(Q_REF)

        self.sim.reset()
        self.init_data = mjx.put_data(self.sim.model, self.sim.data)
        super().__init__(config)  
        self.mx = mjx.put_model(self.sim.model)
        self.sim.reset()
        self.init_data = mjx.put_data(self.sim.model, self.sim.data)
        self.qpos_adr = []
        self.idx_list = []
        for name in self.POLICY_ACTS:
            self.idx_list.append(self.sim.actuator_id(name))
        self.LOWERS, self.UPPERS = self.sim.model.actuator_ctrlrange[jp.array(self.idx_list)].T
        for idx in self.idx_list:
            init_pos = mx.actuator_trnid[idx, 0]
            self.qpos_adr.append(self.sim.model.jnt_qposadr[init_pos])
        self.qpos_adr = np.array(self.qpos_adr)

    @property
    def action_size(self):
        return len(self.POLICY_ACTS)

    @property
    def mj_model(self):
        return self.sim.model

    @property
    def mjx_model(self):
        return self.mx

    @property
    def xml_path(self):
        return XML_PATH
    
    def reset(self, rng):
        dx = dx.replace(qpos=dx.qpos.at[adr_book].set(Q_REF[0]))
        t0 = jax.random.randint(rng, (), 0, len(self.Q_REF) - self._config.episode_length)
        info = {
            "range": rng,
            "step": 0,
            "steps_since_success": 0,
            "last_action": jp.zeros(self.mjx_model.nu),
            "motor_targets": dx.ctrl,
            "qpos_error_history": jp.zerios(1),
            "motor_vel": jp.zeros(6),
            "motor_dir": jp.zeros(6)
        }

        self.obs = self.sim._get_obs(dx, dx.info)
        return dx, t0, Q_REF[0], adr_book, self.obs, info
    
    def step(self,dx, t, target, last_action, action, adr_book, info):
        target = jp.clip(target + action * ACTION_SCALE, self.LOWERS, self.UPPERS)
        dx = dx.replace(ctrl=dx.ctrl.at[self.idx_list].set(target))
        for _ in range(n_substeps):
            dx = step_fn(mx, dx)
        t += 1

        err = dx.qpos[adr_book] - Q_REF[t]
        tracking = jp.exp(-jp.sum(err**2) / tracking_sigma**2)
        info["tracking"] = tracking
        action_rate = jp.sum((action - last_action)**2)
        reward = REWARD_WEIGHTS["tracking"] * tracking + REWARD_WEIGHTS["action_rate"] * action_rate
        print(t, action_rate, reward)
        err_total = bool(jp.linalg.norm(err) > max_tracking_error)
        self.run_result.append([t, dx, target, action])
        if jp.isnan(dx.qpos).any():
            raise ValueError("qpos is NaN")
        if (t >= len(Q_REF) - 1):
            return self.run_result, target, reward, tracking, err_total

            

In [9]:
#smoothing
Q_REF = jp.array(np.clip(Q_REF, LOWERS, UPPERS))
Q_REF = DepthScale.smooth(Q_REF)
print(Q_REF.shape)
elbow = Q_REF[:, 0]
wrist_roll = Q_REF[:, 1]
wrist_tilt = Q_REF[:, 2]
index = Q_REF[:,3]
thumb_add = Q_REF[:,4]
thumb_flex = Q_REF[:,5]

(121, 6)


In [11]:
mx = mjx.put_model(sim.model)
dx = mjx.put_data(sim.model, sim.data)
dx = mjx.step(mx, dx)

mjx.get_data(sim.model, dx)
sid = mujoco.mj_name2id(sim.model, mujoco.mjtObj.mjOBJ_SENSOR, "index_touch")
adr = sim.model.sensor_adr[sid]
dx.sensordata[adr]

step_fn = jax.jit(mjx.step)       # mesh centre vs body origin
adr = sim.model.jnt_qposadr[sim.model.joint("needle_free").id]
sim.data.qpos[adr:adr + 3] = [1,1,1]

sim.set_ctrl("a_index_mcp",0.55)
sim.set_ctrl("a_thumb_pip", -0.05)
sim.set_ctrl("a_thumb_mcp", -0.0233)

mx = mjx.put_model(sim.model)
dx = mjx.put_data(sim.model, sim.data)

for _ in range (1500):
    dx = step_fn(mx, dx)

if np.isnan(dx.qpos).any():
    raise ValueError("NaN in qpos")

sid = mujoco.mj_name2id(sim.model, mujoco.mjtObj.mjOBJ_SENSOR, "index_touch")
sth = mujoco.mj_name2id(sim.model, mujoco.mjtObj.mjOBJ_SENSOR, "thumb_touch")
adr_i = sim.model.sensor_adr[sid]
adr_t = sim.model.sensor_adr[sth]
print(dx.sensordata[adr_i], dx.sensordata[adr_t])

42.66696 42.66696


In [12]:
biggest_jump = []
for i in range(Q_REF.shape[1]):
    fr_diff = []
    fr = 0
    for fr in range (len(Q_REF[:, i]) - 1):
        diff = np.abs(Q_REF[fr, i] - Q_REF[fr+1, i])
        fr_diff.append(diff)
        fr += 1
    biggest_jump.append(max(fr_diff))
    fr_diff = []
ACTION_SCALE = jp.array(np.array(biggest_jump) * 1.5)
print(dict(zip(POLICY_ACTS, ACTION_SCALE.round(3))))

{'a_elbow_pitch': Array(0.11300001, dtype=float32), 'a_forearm_roll': Array(1.427, dtype=float32), 'a_wrist_tilt': Array(0.289, dtype=float32), 'a_index_mcp': Array(0.24000001, dtype=float32), 'a_thumb_mcp': Array(0.033, dtype=float32), 'a_thumb_pip': Array(0., dtype=float32)}


In [13]:
CTRL_DT = 1/30
SIM_DT = 0.002 #timestep in sim
n_substeps = round(CTRL_DT / SIM_DT)
episode_length = 260
EMA_ALPHA = 0.8 #exponential moving average  - smoothing filter
noise = 0.01
tracking_sigma = 0.3 #eval of the copying score
max_tracking_error = 0.8 # maximum error allowance for target to arrive at
REWARD_WEIGHTS = {
    "tracking": 1.0,
    "pinch": 0.0,
    "action_rate": -0.01,
    "termination": -5.0
}


In [ ]:
def main(qpos_adr):
    dx, t0, last_action, base_frame, adr_book, obs, info = reset(qpos_adr, last_action)
    run_result.clear()
    action = np.zeros(6)
    steps = 0
    target = info["motor_targets"]

    while steps < (len(Q_REF) - 1):
        run_result[steps], target, reward, tracking, err_total = step(dx, steps, target, run_result[-1], action, qpos_adr, info)
        steps += 1
        info["step"].append(steps)
    if err_total is True:
        return run_result, dx, t, target, reward, tracking, err_total
    